# text (CLIP) conditioned —— generation & fusion (annotated version)

Load the model trained by `pokemon_train_clip.ipynb`, three ways to play:
1. **Generate by name** (look up the CLIP table cached during training)
2. **Generate by free text** (write any description, e.g. `"a fire-type turtle"`) ← the biggest advantage of text conditioning
3. **Fusion** (SLERP on CLIP vectors)

Self-contained: model/diffusion definitions are inlined (**see `pokemon_train_clip.ipynb` for the detailed explanation**; here only a slimmed-down version is kept for loading).

## Model and diffusion definitions (consistent with the training version, inlined for standalone loading)

In [ ]:
import math, os, json
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, groups=8):
        super().__init__()
        self.block1 = nn.Sequential(nn.GroupNorm(groups, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.cond_mlp = nn.Sequential(nn.SiLU(), nn.Linear(c_dim, out_ch))
        self.block2 = nn.Sequential(nn.GroupNorm(groups, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, c):
        h = self.block1(x); h = h + self.cond_mlp(c)[:, :, None, None]; h = self.block2(h)
        return h + self.skip(x)

class AttnBlock(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch); self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H*W).transpose(1, 2); h, _ = self.mha(h, h, h)
        return x + h.transpose(1, 2).reshape(B, C, H, W)

class Stage(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, n_blocks=2, attn=False):
        super().__init__()
        self.blocks = nn.ModuleList([ResBlock(in_ch if i == 0 else out_ch, out_ch, c_dim) for i in range(n_blocks)])
        self.attn = AttnBlock(out_ch) if attn else None
    def forward(self, x, c):
        for b in self.blocks: x = b(x, c)
        if self.attn is not None: x = self.attn(x)
        return x

class UNet(nn.Module):
    def __init__(self, in_ch=3, base=128, c_dim=256, n_blocks=2, clip_dim=512, clip_table=None):
        super().__init__()
        self.c_dim = c_dim
        self.num_classes = (clip_table.shape[0] - 1) if clip_table is not None else None
        if clip_table is not None: self.register_buffer("clip_table", clip_table)
        self.text_proj = nn.Sequential(nn.Linear(clip_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))
        self.time_mlp  = nn.Sequential(nn.Linear(c_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down1 = Stage(base,   base,   c_dim, n_blocks)
        self.down2 = Stage(base,   base*2, c_dim, n_blocks)
        self.down3 = Stage(base*2, base*4, c_dim, n_blocks, attn=True)
        self.downsample = nn.AvgPool2d(2)
        self.mid = Stage(base*4, base*4, c_dim, n_blocks, attn=True)
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = Stage(base*4 + base*4, base*2, c_dim, n_blocks, attn=True)
        self.up2 = Stage(base*2 + base*2, base,   c_dim, n_blocks)
        self.up1 = Stage(base   + base,   base,   c_dim, n_blocks)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, in_ch, 3, padding=1))
    def cond(self, t, y=None, y_emb=None):
        c = self.time_mlp(sinusoidal_embedding(t, self.c_dim))
        if y_emb is not None:
            c = c + self.text_proj(y_emb)
        elif self.num_classes is not None:
            if y is None:
                y = torch.full((t.size(0),), self.num_classes, device=t.device, dtype=torch.long)
            c = c + self.text_proj(self.clip_table[y])
        return c
    def forward(self, x, t, y=None, y_emb=None):
        c = self.cond(t, y, y_emb); x = self.in_conv(x)
        s1 = self.down1(x, c); x = self.downsample(s1)
        s2 = self.down2(x, c); x = self.downsample(s2)
        s3 = self.down3(x, c); x = self.downsample(s3)
        x = self.mid(x, c)
        x = self.upsample(x); x = self.up3(torch.cat([x, s3], 1), c)
        x = self.upsample(x); x = self.up2(torch.cat([x, s2], 1), c)
        x = self.upsample(x); x = self.up1(torch.cat([x, s1], 1), c)
        return self.out(x)

Sampler `Diffusion.sample`: repeatedly denoises from noise; when conditioned, uses CFG `ε = ε_uncond + w·(ε_cond−ε_uncond)`.

In [ ]:
class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = timesteps; self.device = device
        beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.beta = beta; self.alpha = 1.0 - beta; self.alpha_bar = torch.cumprod(self.alpha, dim=0)
    @torch.no_grad()
    def sample(self, model, n, y=None, y_emb=None, guidance=3.0, img_size=96, channels=3):
        model.eval()
        x = torch.randn(n, channels, img_size, img_size, device=self.device)
        use_cfg = (model.num_classes is not None) and (y is not None or y_emb is not None)
        if use_cfg:
            null = torch.full((n,), model.num_classes, device=self.device, dtype=torch.long)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            if use_cfg:
                eps_c = model(x, t, y=y, y_emb=y_emb); eps_u = model(x, t, y=null)
                pred = eps_u + guidance * (eps_c - eps_u)
            else:
                pred = model(x, t)
            bt, at, abt = self.beta[i], self.alpha[i], self.alpha_bar[i]
            mean = (1/at.sqrt()) * (x - (bt/(1-abt).sqrt()) * pred)
            x = mean + (bt.sqrt()*torch.randn_like(x) if i > 0 else 0.0)
        model.train()
        return x

## Load weights + clip_table + CLIP encoder

- `clip_table`: the CLIP vector table saved during training (generation/fusion by name looks it up)
- `classes.json`: name list (aligned with clip_table)
- Weight ckpt: training saves the **EMA weights**
- Change `EXP_NAME` to point to the experiment you want, and `CKPT` selects the epoch

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

EXP_NAME  = "exp06_stage2_pokemon20aug_clipText"   # ← change to your experiment
OUT_DIR   = os.path.join("/root/autodl-tmp/experiments", EXP_NAME)
CKPT      = os.path.join(OUT_DIR, "checkpoints", "ckpt_ep120.pt")   # ← change to the epoch you want
BASE = 128; N_BLOCKS = 2; IMG_SIZE = 96; TIMESTEPS = 1000

names = json.load(open(os.path.join(OUT_DIR, "classes.json"), encoding="utf-8"))
clip_table = torch.load(os.path.join(OUT_DIR, "clip_table.pt"), map_location=device)
CLIP_DIM = clip_table.shape[1]
model = UNet(base=BASE, n_blocks=N_BLOCKS, clip_dim=CLIP_DIM, clip_table=clip_table).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.eval()
diff = Diffusion(timesteps=TIMESTEPS, device=device)
print("loaded:", EXP_NAME, "| num classes:", len(names))

Load the CLIP text encoder (only serves the "free text" section; the first run downloads ~600MB of weights).

In [ ]:
# CLIP text encoder: only used for "free text" (encodes your input sentence into a vector). Not needed for by-name / fusion.
try:
    import open_clip
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
    import open_clip
_clip, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
_tok = open_clip.get_tokenizer("ViT-B-32"); _clip = _clip.to(device).eval()

@torch.no_grad()
def clip_encode(texts):
    """One or more sentences -> normalized CLIP vectors (N, 512)."""
    if isinstance(texts, str): texts = [texts]
    f = _clip.encode_text(_tok(texts).to(device)).float()
    return f / f.norm(dim=-1, keepdim=True)

## Utilities

- `show`: render the tensor as a grid
- `slerp`: **spherical interpolation**, interpolates between two CLIP vectors while **preserving vector length**.
  Linear interpolation shortens the norm -> goes out of distribution -> generates all white; SLERP travels along the sphere with constant length, so fusion works correctly.

In [ ]:
def show(imgs, title=""):
    imgs = (imgs.clamp(-1,1)+1)/2
    grid = make_grid(imgs.cpu(), nrow=min(len(imgs),4)).permute(1,2,0).numpy()
    plt.figure(figsize=(8,8)); plt.imshow(grid); plt.axis("off"); plt.title(title); plt.show()

def slerp(a, b, t):
    a, b = a.squeeze(0), b.squeeze(0)
    om = torch.acos(((a/a.norm())*(b/b.norm())).sum().clamp(-1,1)); so = torch.sin(om)
    out = ((1-t)*a+t*b) if so < 1e-6 else (torch.sin((1-t)*om)/so)*a + (torch.sin(t*om)/so)*b
    return out.unsqueeze(0)

## 1. Generate by name

Pick one (using training-cached CLIP vectors, looking up `clip_table`). Larger `guidance` fits more closely (3~6 is common).

In [ ]:
NAME = "pikachu"; N = 8; GUIDANCE = 4.0
idx = names.index(NAME)
y = torch.full((N,), idx, device=device, dtype=torch.long)
show(diff.sample(model, N, y=y, guidance=GUIDANCE, img_size=IMG_SIZE), f"{NAME} x{N}")

## 2. Generate by free text (the biggest advantage of text conditioning)

Write any description; CLIP encodes it on the fly into a vector used as the condition (via `y_emb`). You can write **combinations not present in training**,
For example `"a blue fire-type turtle pokémon, upright body"` —— the discrete-label version can't do this.
Keeping the description style close to the training format (name, color, type, body size, generation) works better.

In [ ]:
PROMPT = "a blue fire-type turtle pokémon, upright body, from generation 1"
N = 8; GUIDANCE = 4.0
emb = clip_encode(PROMPT).repeat(N, 1)                 # (N, 512) raw CLIP vector
show(diff.sample(model, N, y_emb=emb, guidance=GUIDANCE, img_size=IMG_SIZE), PROMPT[:40])

## 3. Fuse two (SLERP on CLIP vectors)

Take the CLIP vectors of the two, SLERP interpolate, and get the fused condition. `ALPHA`: 0=pure A, 1=pure B, in between is fusion.
You can also replace the two lines below with `clip_encode("...")` to fuse **two pieces of free text**. If it washes out white, lower `GUIDANCE` to 2~3.

In [ ]:
NAME_A = "charmander"; NAME_B = "squirtle"; ALPHA = 0.5; N = 8; GUIDANCE = 4.0
ea = clip_table[names.index(NAME_A)].unsqueeze(0)
eb = clip_table[names.index(NAME_B)].unsqueeze(0)
fused = slerp(ea, eb, ALPHA).repeat(N, 1)
show(diff.sample(model, N, y_emb=fused, guidance=GUIDANCE, img_size=IMG_SIZE), f"{NAME_A} x {NAME_B} (a={ALPHA})")

## Notes

- **Free text** is the biggest highlight of this version: it can generate semantic combinations not seen in training.
- The closer the description matches the training format, the better the result; CLIP mainly captures strong signals like color/type/body shape.
- Using **Stage 2** weights gives the clearest results for those 20; Stage 1 covers all Pokémon but each is a bit rough.
- Fusion/free text turning white → lower `GUIDANCE` to 2~3 (high guidance amplifies out-of-distribution instability).